## Exercise Embedding

This notebook produces a fixed-size vector representation for each exercise in the MIAAM dataset.

**Pipeline:**
1. Load `maths_exercises_table.parquet` (~7 151 exercises).
2. Filter out the 12 exercises that have no compressed screenshot, leaving **6 936 exercises**.
3. Encode each exercise with [`SmolVLM-256M-Instruct`](https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct) — a multimodal VLM that jointly processes the exercise **screenshot** and its **text metadata** (title, description, …). The last hidden state is pooled into a single `Float32` vector.
4. Save the result to `data/exercises_embedded.parquet` with columns `exercise_id` (string) and `embedding` (fixed-length float array).

**Output:** one embedding per exercise, suitable for similarity search, clustering, or as input features for downstream models.


In [1]:

import polars as pl
import json
import pathlib
import os
import numpy as np
from exercise_embedding import encode_all
from attempt_embedding import encode_interactions
from tqdm import tqdm

DATASET = pathlib.Path("../../MIAAM")
SAVE_FOLDER = pathlib.Path("../data")

exercises    = pl.read_parquet(DATASET / "data/maths_exercises_table.parquet")

/home/guglielmo/Projects/stundetsTracker/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Filter exercises without a screenshot

Some exercises in `maths_exercises_table.parquet` have no corresponding screenshot in `data/screenshots/compressed/`. They are removed from both `exercises` and `interactions` before any further processing, so that all remaining exercises have visual content available for multimodal models.

In [ ]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}

missing_screenshots = exercises.filter(~pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"Exercises without a raw screenshot: {len(missing_screenshots)}")
print(missing_screenshots.select(["exercise_id", "source", "module_name", "activity_name"]))

exercises    = exercises.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))

In [ ]:
exercises_embedding = encode_all(exercises, SCREENSHOTS)
exercises_embedding.write_parquet(SAVE_FOLDER / "exercises_embedded.parquet")


In [2]:
import polars as pl

exercises_embedding = pl.read_parquet("../data/exercises_embedded.parquet")

In [3]:
exercises_embedding.head()

exercise_id,embedding
str,"array[f32, 576]"
"""39676249-728a-4d21-9a58-c31451…","[-0.625, -1.140625, … -1.898438]"
"""1c921f06-1005-11ed-861d-0242ac…","[-1.132812, -2.5625, … -1.515625]"
"""1a269758-11b6-11ed-861d-0242ac…","[-0.894531, -2.015625, … -1.617188]"
"""c5759d6c-ebc2-11ec-8ea0-0242ac…","[-0.90625, -2.140625, … -3.09375]"
"""b9dbda26-a4bb-48ae-825b-a33b08…","[-0.828125, -1.117188, … -1.367188]"
